# LlamaIndex: RAG-конвейер для вопросов по собственным документам

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

LlamaIndex — фреймворк для построения систем **RAG** (Retrieval-Augmented Generation, генерация с опорой на поиск). Он решает задачу, с которой сталкивается каждый исследователь: языковая модель «не знает» ваши рукописи, препринты, лабораторные журналы и внутренние отчёты. LlamaIndex соединяет ваши документы и языковую модель в единый конвейер: сначала находит фрагменты, релевантные вопросу, затем передаёт их модели как контекст и получает ответ со ссылкой на источник.

Для исследователя это значит: можно задавать вопросы на естественном языке по всему архиву своих статей и получать ответы с точным указанием того, из какой работы взят каждый факт. Риск «выдуманных» ответов снижается, потому что модель опирается не на обученные параметры, а на конкретные найденные абзацы.

В этом блокноте мы пройдём полный цикл: текстовые файлы → документы LlamaIndex → узлы (чанки) → векторный индекс → движок запросов → ответ с цитированием. Расчётное время прохождения — около пяти–семи минут. Блокнот не требует API-ключей и платных сервисов.

## Ключевые понятия LlamaIndex

| Понятие | Что это | Аналогия |
|---|---|---|
| **Document** | Один исходный документ: текстовый файл, PDF, веб-страница | Статья в папке |
| **Node / чанк** | Фрагмент документа фиксированного размера (обычно 256–1024 токена) с метаданными | Абзац со сноской на статью |
| **VectorStoreIndex** | Индекс, хранящий эмбеддинги всех узлов; поддерживает семантический поиск | Предметный указатель с векторами |
| **Retriever** | Компонент, принимающий вопрос и возвращающий топ-K наиболее близких узлов | Референт библиотеки |
| **QueryEngine** | Оркестратор: вызывает Retriever, формирует промпт с найденным контекстом и получает ответ от LLM | Научный ассистент |
| **SourceNode** | Узел с оценкой релевантности, возвращаемый вместе с ответом | Список ссылок к ответу |
| **Settings** | Глобальный конфигурационный объект: какую модель эмбеддингов и какую LLM использовать | Настройки рабочего места |

Ключевое отличие от «голого» семантического поиска (см. блокнот «Большие языковые модели»): LlamaIndex не просто ранжирует документы, а передаёт найденные фрагменты в LLM и получает связный ответ — с явным указанием, из какого узла взят каждый факт.

## 1. Установка библиотек

Устанавливаем модульные пакеты LlamaIndex. Начиная с версии 0.10, каждый интеграционный компонент — отдельный пакет; это позволяет не тянуть лишние зависимости.

- `llama-index-core` — ядро фреймворка (Document, Node, VectorStoreIndex, QueryEngine).
- `llama-index-embeddings-huggingface` — локальные эмбеддинги через HuggingFace Transformers (без API-ключей).
- `llama-index-llms-huggingface` — локальный запуск небольших instruct-моделей через HuggingFace (без API-ключей).

Платных зависимостей нет. OpenAI не устанавливается и не вызывается.

In [ ]:
%pip install -q \
    llama-index-core \
    llama-index-embeddings-huggingface \
    llama-index-llms-huggingface \
    sentence-transformers \
    transformers \
    accelerate \
    torch

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`; вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Создадим пять небольших текстовых файлов, имитирующих фрагменты научных статей из разных областей. Это позволяет показать реальный путь «файлы → SimpleDirectoryReader → Document», который используют исследователи с собственным архивом.

Каждый файл — несколько абзацев по одной теме. LlamaIndex разобьёт их на узлы и сохранит имя файла как метаданные источника.

In [ ]:
import os
import pathlib

# Директория для временных файлов статей
DOCS_DIR = pathlib.Path("demo_papers")
DOCS_DIR.mkdir(exist_ok=True)

# Пять имитированных фрагментов научных статей
papers = {
    "трансформеры_в_биоинформатике.txt": """\
Трансформерные архитектуры в биоинформатике
Модели на основе механизма внимания (attention) произвели переворот в обработке
биологических последовательностей. Архитектура трансформера, предложенная Vaswani
et al. (2017), позволяет моделировать дальние зависимости между элементами
последовательности без рекуррентных вычислений.

Применительно к белковым последовательностям это дало модели ESM-2 (Lin et al.,
2023) способность предсказывать трёхмерную структуру белка по первичной
последовательности аминокислот. Модель обучена на UniRef50 — базе из более 250
миллионов уникальных белковых последовательностей.

Метод PocketCrafter, разработанный в 2024 году, использует трансформерные
представления белков для предсказания потенциальных лиганд-связывающих карманов,
что существенно ускоряет рациональный дизайн лекарств на стадии виртуального
скрининга. Точность по метрике DCC достигает 71% на наборе COACH-420.
""",

    "графовые_сети_для_молекул.txt": """\
Графовые нейронные сети для молекулярного моделирования
Молекула естественным образом представима как граф: атомы — вершины, ковалентные
связи — рёбра. Графовые нейронные сети (GNN) напрямую работают с такой структурой,
не требуя ручного конструирования признаков (дескрипторов).

Архитектура SchNet (Schütt et al., 2017) является непрерывно-фильтровой свёрточной
нейронной сетью по графу, оперирующей межатомными расстояниями. Она предсказывает
энергии молекулярных конфигураций с точностью, сравнимой с методами квантовой химии
уровня DFT, при скорости инференса на несколько порядков выше.

Модель DimeNet++ добавляет угловые зависимости между связями, что повышает точность
предсказания таких свойств, как дипольный момент и поляризуемость. На наборе QM9
средняя абсолютная ошибка предсказания нулевой точки энергии (ZPVE) составляет
менее 1.5 мэВ.
""",

    "диффузионные_модели_в_химии.txt": """\
Диффузионные модели для генерации молекул
Диффузионные генеративные модели, первоначально разработанные для синтеза
изображений, адаптированы к задаче de novo дизайна молекул. В отличие от вариационных
автоэнкодеров и генеративно-состязательных сетей, они демонстрируют более стабильное
обучение и высокое разнообразие генерируемых образцов.

Модель EDM (Hoogeboom et al., 2022) выполняет диффузию непосредственно в непрерывном
трёхмерном пространстве атомных координат, сохраняя инвариантность относительно
перемещений, поворотов и зеркальных отражений молекулы.

Для условной генерации (с заданным набором целевых свойств) разработан метод GDSS,
объединяющий граф-диффузию по топологии молекулы с диффузией по непрерывным
признакам атомов. Это открывает путь к направленному синтезу молекул с
программируемыми физико-химическими свойствами.
""",

    "климатические_модели_и_ии.txt": """\
Применение машинного обучения в климатическом моделировании
Физические модели общей циркуляции атмосферы (GCM) требуют огромных вычислительных
ресурсов и не могут воспроизводить процессы субсеточного масштаба (конвекция,
облакообразование) напрямую — они параметризуются полуэмпирическими схемами.

Нейросетевые параметризации, обученные на данных высокоразрешённых симуляций
или наблюдений, предлагают замену традиционным схемам. Работа Rasp et al. (2018)
показала, что простая полносвязная сеть, обученная на облачно-разрешающих
симуляциях, точнее воспроизводит вертикальный перенос тепла и влаги, чем
стандартная параметризация.

Подход FourCastNet (Pathak et al., 2022) применяет Фурье-нейронный оператор для
прогноза атмосферных полей с разрешением 0.25 градуса на двое суток вперёд
со скоростью на несколько порядков выше традиционных моделей численного
прогноза погоды.
""",

    "эмбеддинги_для_научных_текстов.txt": """\
Векторные представления для научной литературы
Задача поиска по научным публикациям требует моделей, обученных на специализированном
доменном языке. Общеязыковые модели (BERT, RoBERTa) хуже справляются с нотацией
формул, аббревиатурами и терминологией конкретных дисциплин.

SciBERT (Beltagy et al., 2019) — трансформерная модель, дообученная на 1,14 млн
полных научных статей из Semantic Scholar, охватывающих биомедицину и компьютерные
науки. На задаче называния сущностей (NER) в биомедицинских текстах SciBERT
превосходит BERT-base на 2.6 процентных пункта F1.

SPECTER2 (Singh et al., 2023) обучен задаче метрического обучения по цитированию:
цитируемые друг другом статьи должны иметь близкие векторы. Это делает его особенно
точным для задач: поиск похожих работ, построение графа цитирований, кластеризация
направлений исследований.
""",
}

for filename, content in papers.items():
    (DOCS_DIR / filename).write_text(content, encoding="utf-8")

print(f"Создано файлов: {len(papers)}")
for f in sorted(DOCS_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name}  ({size} байт)")

## 4. Применение инструмента: сквозной RAG-конвейер

Следующие шаги воспроизводят полный рабочий цикл LlamaIndex. Каждый шаг соответствует одному ключевому понятию из раздела «Ключевые понятия» выше.

### Шаг 4.1. Глобальная конфигурация: Settings

Первое, что нужно сделать при работе с LlamaIndex, — явно задать модель эмбеддингов и языковую модель через объект `Settings`. Если этого не сделать, LlamaIndex попытается использовать OpenAI по умолчанию и упадёт с ошибкой об отсутствующем ключе.

Здесь мы задаём:
- **Эмбеддинги** — `sentence-transformers/all-MiniLM-L6-v2`: компактная (22 млн параметров), многоязычная, хорошо работает на смешанных русско-английских научных текстах.
- **LLM** — `MockLLM`: заглушка-имитатор. Шаг поиска (Retrieval) и извлечения узлов работает полностью офлайн и не требует LLM; шаг генерации текстового ответа вынесен отдельно и может быть заменён на реальную модель (см. Шаг 4.4).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM

# --- Эмбеддинги: локально, без API-ключей ---
# Модель загружается с HuggingFace Hub (~90 МБ); при повторных запусках берётся из кэша.
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=512,
)

# --- LLM: MockLLM гарантирует, что OpenAI не вызывается ---
# Ответы-заглушки позволяют показать структуру конвейера.
# Замените на HuggingFaceLLM (см. Шаг 4.4) для реальной генерации.
mock_llm = MockLLM(max_tokens=256)

# Применяем глобально — все компоненты LlamaIndex используют эти объекты
Settings.embed_model = embed_model
Settings.llm = mock_llm

# Дополнительные параметры индексирования
Settings.chunk_size = 512        # максимальный размер чанка в токенах
Settings.chunk_overlap = 64      # перекрытие соседних чанков

print("Конфигурация LlamaIndex:")
print(f"  Модель эмбеддингов : {embed_model.model_name}")
print(f"  LLM                : {type(mock_llm).__name__} (заглушка, OpenAI не используется)")
print(f"  Размер чанка       : {Settings.chunk_size} токенов")
print(f"  Перекрытие чанков  : {Settings.chunk_overlap} токенов")

### Шаг 4.2. Загрузка документов и разбивка на узлы (чанки)

`SimpleDirectoryReader` рекурсивно читает все файлы из директории и создаёт объекты `Document`. Каждый документ хранит текст, путь к файлу и другие метаданные.

Затем `SentenceSplitter` разбивает документы на узлы (`Node`) — перекрывающиеся фрагменты фиксированного размера. Узлы — минимальная единица индексирования: именно они индексируются и возвращаются как «источник» при ответе на вопрос.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

# Шаг A: загружаем документы из директории
documents = SimpleDirectoryReader(
    input_dir=str(DOCS_DIR),
    encoding="utf-8",
).load_data()

print(f"Загружено документов: {len(documents)}")
for doc in documents:
    # file_name находится в метаданных документа
    fname = doc.metadata.get("file_name", "неизвестен")
    print(f"  Документ '{fname}'  —  {len(doc.text)} символов")

# Шаг Б: разбиваем на узлы (чанки) с сохранением ссылки на исходный документ
splitter = SentenceSplitter(
    chunk_size=Settings.chunk_size,
    chunk_overlap=Settings.chunk_overlap,
)
nodes = splitter.get_nodes_from_documents(documents)

print(f"\nВсего узлов (чанков): {len(nodes)}")
print("\nПервые два узла:")
for node in nodes[:2]:
    src = node.metadata.get("file_name", "?")
    preview = node.get_content()[:120].replace("\n", " ")
    print(f"  [{src}]  {preview}...")

### Шаг 4.3. Построение векторного индекса

`VectorStoreIndex.from_documents` выполняет две операции подряд:
1. Вычисляет эмбеддинг каждого узла через настроенную модель (`all-MiniLM-L6-v2`).
2. Сохраняет пары (эмбеддинг, текст узла) во внутреннем векторном хранилище (по умолчанию — в памяти).

После этого индекс готов отвечать на семантические запросы.

In [ ]:
from llama_index.core import VectorStoreIndex

# Строим индекс из уже созданных узлов.
# show_progress=True выводит прогресс-бар в Colab.
index = VectorStoreIndex(
    nodes,
    show_progress=True,
)

print(f"\nИндекс построен.")
print(f"Узлов в индексе: {len(index.docstore.docs)}")
print("Тип хранилища  : SimpleVectorStore (in-memory, данные не покидают машину)")

### Шаг 4.4. Движок запросов и поиск с цитированием источников

`as_retriever` создаёт Retriever — компонент, который по вопросу возвращает топ-K наиболее близких узлов с оценкой релевантности. Этого достаточно, чтобы увидеть ключевую функцию RAG: **ответ со ссылкой на источник**.

Далее показаны два варианта ответа на вопрос:

**Вариант A (всегда работает офлайн):** используем Retriever напрямую — получаем список узлов с оценками и именами файлов-источников.

**Вариант Б (опционально):** используем `as_query_engine` с реальной LLM для генерации текстового ответа. Ячейка защищена блоком `try/except` — если HuggingFace LLM недоступна или не помещается в GPU, конвейер не падает.

> **Внимание:** этот шаг скачивает модель TinyLlama-1.1B (~2 ГБ).

In [ ]:
import textwrap

# --- Вариант A: Retriever (работает полностью офлайн) ---
# Возвращает узлы с оценками релевантности — без вызова LLM.

QUERY = "Какие нейросетевые методы используются для предсказания структуры белка?"

retriever = index.as_retriever(similarity_top_k=3)
source_nodes = retriever.retrieve(QUERY)

print(f"Запрос: {QUERY}")
print(f"\nНайдено релевантных узлов: {len(source_nodes)}")
print("=" * 70)

for rank, node_with_score in enumerate(source_nodes, start=1):
    node  = node_with_score.node
    score = node_with_score.score
    src   = node.metadata.get("file_name", "неизвестен")
    text  = node.get_content()

    print(f"\n[{rank}] Источник: {src}")
    print(f"    Оценка релевантности: {score:.4f}")
    print(f"    Фрагмент:")
    # Выводим первые 300 символов фрагмента с отступом
    wrapped = textwrap.fill(text[:300], width=65, initial_indent="    ",
                            subsequent_indent="    ")
    print(wrapped)
    if len(text) > 300:
        print("    [...]")

In [ ]:
# --- Вариант Б: QueryEngine с реальной LLM (опционально) ---
# Если ресурсов Colab достаточно, используется небольшая instruct-модель
# TinyLlama-1.1B-Chat-v1.0 (~2 ГБ). Она помещается в бесплатный Colab T4.
# При нехватке памяти или ошибке загрузки блок переходит в режим
# MockLLM — результаты поиска остаются доступными.

# Фиксируем зерно для воспроизводимости генерации (do_sample=True)
import torch
torch.manual_seed(42)

try:
    from llama_index.llms.huggingface import HuggingFaceLLM
    from llama_index.core import PromptTemplate

    # Системный промпт для instruct-модели TinyLlama (ChatML-формат)
    SYSTEM_PROMPT = (
        "Ты научный ассистент. Отвечай на вопрос строго по предоставленному "
        "контексту. Если ответа в контексте нет, так и скажи."
    )

    query_wrapper_prompt = PromptTemplate(
        "<|system|>\n" + SYSTEM_PROMPT + "\n</s>\n"
        "<|user|>\n{query_str}\n</s>\n"
        "<|assistant|>\n"
    )

    hf_llm = HuggingFaceLLM(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        query_wrapper_prompt=query_wrapper_prompt,
        context_window=2048,
        max_new_tokens=256,
        generate_kwargs={"temperature": 0.1, "do_sample": True},
        device_map="auto",      # автоматически выбирает GPU/CPU
    )

    # Применяем реальную LLM только для QueryEngine
    Settings.llm = hf_llm

    query_engine = index.as_query_engine(
        similarity_top_k=3,
        response_mode="compact",
    )
    response = query_engine.query(QUERY)

    print("=== Ответ QueryEngine с реальной LLM ===")
    print(f"\nВопрос: {QUERY}")
    print(f"\nОтвет:\n{response.response}")
    print("\n--- Источники ответа ---")
    for i, sn in enumerate(response.source_nodes, 1):
        src   = sn.node.metadata.get("file_name", "?")
        score = sn.score
        print(f"  [{i}] {src}  (релевантность: {score:.4f})")

except Exception as exc:
    # Если модель не загрузилась — откатываемся к MockLLM и сообщаем причину
    Settings.llm = mock_llm
    print(f"HuggingFace LLM недоступна: {exc}")
    print()
    print("Используем MockLLM — шаг поиска (retrieval) из Варианта A работает полноценно.")
    print("Результаты поиска с цитированием источников выведены в ячейке выше.")
    print()
    print("Чтобы получить связный текстовый ответ без GPU, можно:")
    print("  1. Использовать Colab с T4-GPU (меню Среда выполнения → Сменить тип среды).")
    print("  2. Подключить GigaChat / YandexGPT через соответствующие LlamaIndex-интеграции.")
    print("  3. Запустить Ollama локально и подключить через llama-index-llms-ollama.")

### Шаг 4.5. Несколько дополнительных запросов

Проверим конвейер на трёх разных вопросах, чтобы показать, что индекс корректно маршрутизирует запросы к тематически релевантным документам.

In [ ]:
additional_queries = [
    "Как граф-сети предсказывают молекулярные свойства?",
    "Какие модели применяют для прогноза погоды с помощью ИИ?",
    "Что такое SPECTER и для чего он используется в науке?",
]

print("Результаты дополнительных запросов")
print("=" * 70)

for q in additional_queries:
    hits = retriever.retrieve(q)
    print(f"\nЗапрос: {q}")
    for rank, nws in enumerate(hits[:2], 1):
        src   = nws.node.metadata.get("file_name", "?")
        score = nws.score
        snippet = nws.node.get_content()[:100].replace("\n", " ")
        print(f"  [{rank}] {src}  (score={score:.4f})")
        print(f"       «{snippet}...»")

## 5. Интерпретация результатов

### Оценки релевантности узлов

График ниже показывает оценки косинусного сходства для узлов, найденных по основному запросу. Это позволяет наглядно понять, насколько уверенно индекс выбрал источники.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Собираем оценки по всем узлам индекса для основного запроса
retriever_all = index.as_retriever(similarity_top_k=len(nodes))
all_hits = retriever_all.retrieve(QUERY)

# Сортируем по убыванию оценки
all_hits_sorted = sorted(all_hits, key=lambda x: x.score, reverse=True)

labels  = []
scores  = []
sources = []
unique_sources = sorted({
    h.node.metadata.get("file_name", "?") for h in all_hits_sorted
})
# Цвет по источнику
src_color = {src: VIZ["series"][i % len(VIZ["series"])]
             for i, src in enumerate(unique_sources)}

for nws in all_hits_sorted:
    src = nws.node.metadata.get("file_name", "?")
    # Короткий ярлык: первые 25 символов имени файла
    short = src[:25].replace("_", " ").replace(".txt", "")
    labels.append(short)
    scores.append(nws.score)
    sources.append(src)

fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.38)))

bar_colors = [src_color[s] for s in sources]
bars = ax.barh(
    range(len(labels)),
    scores,
    color=bar_colors,
    edgecolor=VIZ["edge"],
    linewidth=0.7,
)

# Пунктирная линия — порог top-3
top3_threshold = all_hits_sorted[2].score if len(all_hits_sorted) >= 3 else 0
ax.axvline(top3_threshold, color=VIZ["series"][2], linestyle="--",
           linewidth=1.4, label=f"Порог топ-3 ({top3_threshold:.3f})")

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Косинусная близость к запросу")
ax.set_title(
    f"Релевантность узлов индекса\n«{QUERY[:55]}...»",
    pad=12,
)

# Легенда по источникам
patches = [mpatches.Patch(color=c, label=src.replace("_", " ").replace(".txt", ""))
           for src, c in src_color.items()]
ax.legend(handles=patches, loc="lower right", fontsize=8,
          title="Источник", title_fontsize=9)

fig.tight_layout()
plt.show()

print(f"\nИнтерпретация:")
print(f"  Узлов с оценкой выше порога топ-3 ({top3_threshold:.3f}): "
      f"{sum(1 for s in scores if s >= top3_threshold)}")
print(f"  Наиболее релевантный источник: "
      f"{all_hits_sorted[0].node.metadata.get('file_name', '?')}")

### Как читать результаты конвейера

**Оценка релевантности (score)** — косинусная близость между вектором запроса и вектором узла. Значения близки к 1, если смысл запроса и фрагмента совпадают; близки к 0, если тексты на разные темы. Отрицательных значений при нормированных векторах нет.

**Цитирование источника** (`source_nodes`) — главное практическое преимущество LlamaIndex перед свободной генерацией. Каждый факт в ответе можно проследить до конкретного файла и фрагмента. В реальном проекте `file_name` заменяется на DOI, номер страницы или любые другие метаданные.

**Порог top-K** — параметр `similarity_top_k` балансирует полноту (больше K — больше контекста) и точность (меньше K — отфильтрованы шумные фрагменты). При коротких документах K=2–3 обычно оптимально; при длинных архивах K=5–10.

**Когда оценки у всех узлов одинаково низкие:** вопрос сформулирован на языке, которого нет в документах, или коллекция не содержит релевантной информации. Это честный сигнал — RAG не выдумывает ответ, а возвращает то, что есть.

**Когда ответ некорректен:** проблема обычно в Retriever (плохо выбраны фрагменты), а не в LLM. Проверьте `source_nodes` — если они нерелевантны, увеличьте чанки, смените модель эмбеддингов или добавьте фильтрацию по метаданным.

## 6. Попробуйте на своих данных

Замените демонстрационные файлы своим архивом статей. Типичный сценарий: выгрузить PDF из Zotero, положить в одну папку и запустить блокнот заново.

1. Загрузите файлы в Colab: панель слева → «Файлы» → кнопка загрузки.
2. Для PDF установите дополнительный парсер и укажите свою директорию (см. комментарии ниже).
3. Измените запрос `MY_QUERY` на свой научный вопрос.
4. Выполните блокнот заново — индекс пересчитается автоматически.

In [ ]:
# =============================================================
# Шаблон для работы со своими документами
# =============================================================

# --- 1. Укажите путь к вашей директории с файлами ---
# MY_DOCS_DIR = "/content/my_papers"       # PDF и .txt — Colab
# MY_DOCS_DIR = "/content/drive/MyDrive/papers"  # Google Drive

# --- 2. Для PDF необходим дополнительный парсер pypdf ---
# !pip install -q pypdf
# from llama_index.core import SimpleDirectoryReader
# my_documents = SimpleDirectoryReader(
#     input_dir=MY_DOCS_DIR,
#     required_exts=[".pdf", ".txt"],   # укажите нужные расширения
# ).load_data()

# --- 3. Добавьте метаданные вручную (опционально) ---
# for doc in my_documents:
#     doc.metadata["doi"] = "10.XXXX/..."   # можно добавить DOI
#     doc.metadata["year"] = 2024

# --- 4. Постройте индекс и задайте свой вопрос ---
# my_index = VectorStoreIndex.from_documents(my_documents, show_progress=True)
# my_retriever = my_index.as_retriever(similarity_top_k=3)

# MY_QUERY = "Ваш научный вопрос на русском или английском языке"
# my_hits = my_retriever.retrieve(MY_QUERY)
# for rank, nws in enumerate(my_hits, 1):
#     src = nws.node.metadata.get("file_name", "?")
#     print(f"[{rank}] {src}  score={nws.score:.4f}")
#     print(f"  {nws.node.get_content()[:200]}...")

print("Шаблон готов. Раскомментируйте нужные строки и укажите путь к своим файлам.")

## 7. Что дальше

**Документация и ресурсы:**
- Официальный сайт LlamaIndex: https://www.llamaindex.ai/
- Документация (актуальная, модульные пакеты v0.10+): https://docs.llamaindex.ai/
- Список всех интеграций (LLM, векторные хранилища, загрузчики): https://llamahub.ai/

**Следующие шаги в реальном проекте:**

1. **Постоянный индекс.** Добавьте `ChromaVectorStore` или `QdrantVectorStore`, чтобы не пересчитывать эмбеддинги при каждом запуске. Векторное хранилище сохраняется на диск или в облако.

2. **Оценка качества (RAG Evaluation).** LlamaIndex предоставляет `FaithfulnessEvaluator` и `RelevancyEvaluator` для автоматической оценки, насколько ответ опирается на найденные фрагменты.

3. **Иерархический индекс.** Для больших коллекций (тысячи статей) используйте `SummaryIndex` над документами и `VectorStoreIndex` над чанками — двухуровневый поиск ускоряет навигацию.

4. **Метаданные как фильтры.** Добавьте к документам год, автора, журнал — и фильтруйте по ним на уровне Retriever: `retriever = index.as_retriever(filters=MetadataFilters(...))`.

5. **Подключение реальной LLM.** Без GPU: GigaChat и YandexGPT через их API (`llama-index-llms-gigachat`, `llama-index-llms-yandexgpt`). С GPU: любая HuggingFace instruct-модель через `HuggingFaceLLM` из Шага 4.4.

6. **Агентный поиск.** `OpenAIAgent` / `ReActAgent` позволяет LlamaIndex последовательно уточнять запросы и объединять результаты из нескольких индексов — для сложных многошаговых вопросов.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике релевантности все узлы из файла `трансформеры_в_биоинформатике.txt` получили высокую оценку по запросу о предсказании структуры белка, а узлы из `климатические_модели_и_ии.txt` — близкую к нулю. Что именно обеспечивает такое разделение, и изменится ли результат, если задать тот же запрос на английском языке?
2. Исследователь получил ответ от QueryEngine и собирается процитировать его в статье без проверки исходных фрагментов. Назовите конкретный риск, который присутствует даже при RAG-подходе, и как его снизить, не меняя архитектуру конвейера.
3. Вы загружаете архив из 2 000 PDF-статей и замечаете, что при `similarity_top_k=3` ответы часто пропускают релевантные фрагменты из длинных статей. Назовите два параметра конвейера, которые стоит скорректировать, и объясните, как каждый из них влияет на полноту поиска.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Нейросетевая модель эмбеддингов сопоставила смысл запроса и содержимое узлов в общем векторном пространстве: понятия «предсказание структуры белка», «трансформеры», «ESM-2» оказались геометрически близки, а «прогноз погоды» и «GCM» — далеко. Многоязычная модель `all-MiniLM-L6-v2` обрабатывает русский и английский в одном пространстве, поэтому английский запрос найдёт те же русскоязычные фрагменты.
2. Даже при RAG языковая модель может неверно синтезировать или обобщить найденные фрагменты, искажая смысл оригинала, — это синтетическая галлюцинация. Снизить риск без смены архитектуры: всегда проверять `source_nodes` вручную и цитировать исходный текст фрагмента, а не сгенерированный ответ.
3. Первый параметр — `chunk_size`: при слишком малом размере чанка длинный релевантный абзац разрезается, и его части получают меньшую оценку, чем целый фрагмент. Увеличение `chunk_size` сохраняет больше контекста в одном узле. Второй параметр — `similarity_top_k`: увеличение с 3 до 5–10 повышает полноту (recall), позволяя попасть в выборку большему числу релевантных фрагментов за счёт некоторого снижения точности.
</details>
